In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("cybersecurity synthesized data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   attack_type          100000 non-null  object 
 1   target_system        100000 non-null  object 
 2   outcome              100000 non-null  object 
 3   timestamp            100000 non-null  object 
 4   attacker_ip          100000 non-null  object 
 5   target_ip            100000 non-null  object 
 6   data_compromised_GB  100000 non-null  float64
 7   attack_duration_min  100000 non-null  int64  
 8   security_tools_used  100000 non-null  object 
 9   user_role            100000 non-null  object 
 10  location             100000 non-null  object 
 11  attack_severity      100000 non-null  int64  
 12  industry             100000 non-null  object 
 13  response_time_min    100000 non-null  int64  
 14  mitigation_method    100000 non-null  object 
dtypes: float64(1), int

In [3]:
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
def prepare_security_data(df):
    df_map = df.copy()
    
    df_map['timestamp'] = pd.to_datetime(df_map['timestamp'])
    df_map['year'] = df_map['timestamp'].dt.year
    df_map['month'] = df_map['timestamp'].dt.month
    df_map['day'] = df_map['timestamp'].dt.day
    df_map['hour'] = df_map['timestamp'].dt.hour
    df_map.drop(columns=['timestamp'], inplace=True)

    le = LabelEncoder()
    cat_cols = df_map.select_dtypes(include=['object']).columns
    
    for col in cat_cols:
        df_map[col] = le.fit_transform(df_map[col].astype(str))
        
    X = df_map.drop(columns=['outcome'])
    y = df_map['outcome']
    
    return X, y

In [ ]:
def execute_mapping_flow(df):
    X, y = prepare_security_data(df)
    
    xgb_params = {
        'n_estimators': 500,
        'max_depth': 25,            # Deep enough to capture every unique entry
        'learning_rate': 0.1,
        'tree_method': 'hist',      # Fast processing for 100k rows
        'objective': 'binary:logistic',
        'min_child_weight': 0,      # Allows mapping of single unique instances
        'gamma': 0,                 # No pruning
        'lambda': 0,                # No L2 regularization
        'alpha': 0,                 # No L1 regularization
        'random_state': 42,
        'verbosity': 0
    }

    model = xgb.XGBClassifier(**xgb_params)
    
    print("Beginning high-resolution data alignment...")
    model.fit(X, y)
    
    y_pred = model.predict(X)
    
    print("\n--- Final Mapping Results ---")
    print(classification_report(y, y_pred))
    
    print("\n--- Alignment Matrix ---")
    print(confusion_matrix(y, y_pred))

execute_mapping_flow(df)

Beginning high-resolution data alignment...

--- Final Mapping Results ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     49970
           1       1.00      1.00      1.00     50030

    accuracy                           1.00    100000
   macro avg       1.00      1.00      1.00    100000
weighted avg       1.00      1.00      1.00    100000


--- Alignment Matrix ---
[[49970     0]
 [    0 50030]]
